# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: d:\SOCASIS\Ingineria AI\echochamber-project-team-4
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student 3"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [7]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [8]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [11]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [14]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(10, random_state=35)

,source_channel,video_title,text
27140,turcescu111,"Țoiu MINTE, refuză să răspundă și fuge din con...",Felicitări jurnaliștilor care nu s-au lăsat ma...
29490,otvdirect,"DIANA SOSOACA, MARIAN VANGHELIE IN DIRECT|OTV ...",Stimă și respect ptr Doamna Europarlamentar Di...
8795,RecorderRomania,Judecătoarea Raluca Moroșanu: „Dacă nu se schi...,Nu înțeleg de ce se mira oameni în halul asta ...
14647,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,Am asistat la un moment dat la un incident car...
18512,RecorderRomania,PORTRET DE CANDIDAT: Crin Antonescu,Sunteți cei mai buni ! Va multumim ca ne desch...
3923,digi24hd56,Scandal la Spitalul Județean Constanța. Neuroc...,"Daca nu vrea , nu este obligat sa facă gărzi. ..."
12469,RecorderRomania,DOCUMENTAR RECORDER. Singuri,"Prea dur, mi se face rău, nu pot termina docum..."
7819,RecorderRomania,Raiul evazioniștilor. Investigație din interio...,Victor Nica - tupeu de borfaș (disappointed bu...
14480,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,Sunteți bineveniți în Alba Iulia să vedeți cum...
22780,CălinGeorgescu-CanalulOficial,Călin Georgescu - Cad zidurile minciunii ( 10....,"Drepturile și libertățile se iau, nu se dau! R..."


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [15]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [17]:
# Prompt de sistem: definește rolul modelului ca un critic al curentelor suveraniste
SYSTEM_PROMPT = """Ești un analist politic pro-european și ferm anti-suveranist. 
Rolul tău este să analizezi comentariile politice, demascând manipulările populiste, retorica anti-UE și naționalismul izolaționist. 
Ești incisiv în a identifica modul în care suveranismul subminează stabilitatea democratică și valorile occidentale.
Răspunzi concis, folosind un limbaj academic dar critic la adresa curentelor extremiste.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și extrage o analiză riguroasă bazată pe următoarele criterii:

1. target: Identifică entitatea vizată (ex: UE, NATO, Guvern, un politician anume sau o instituție democratică).
2. stance: Determină poziția autorului față de valorile europene (Sprijin, Opoziție suveranistă, Neutru).
3. sentiment: Evaluează starea emoțională a textului pe o scară de la "Extrem de ostil" la "Pozitiv".
4. tone: Caracterizează stilul comunicării (ex: Populist, Agresiv, Rațional, Conspiraționist).
5. topic: Identifică tema principală (ex: Suveranitate națională, Economie, Migrație, Securitate).
6. interpretation_problem: Explică succat de ce mesajul este problematic din punct de vedere anti-suveranist (ex: promovează dezinformarea, instigă la izolaționism, folosește limbaj radical).

Important:
Analiza trebuie să prioritizeze identificarea elementelor care pun în pericol coeziunea europeană.
Returnează EXCLUSIV un obiect JSON valid cu exact aceste chei:
"target", "stance", "sentiment", "tone", "topic", "interpretation_problem".

Comentariu de analizat:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [22]:
from openai import OpenAI
client = OpenAI(
api_key=os.getenv("GEMINI_API_KEY"), 
base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [23]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [26]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""AUR (Alianța pentru U..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion, instit..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Guvernul României / I..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""UE, Guvernul României..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""Instituții democratic..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George (un politician..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""Uniunea Europeană (im..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""AUR (Alianța pentru U..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George (un politician..."


# 9. Verificam rezultatele

In [29]:
results_df.model_output[3]

'```json\n{\n  "target": "Guvernul României / Instituțiile responsabile cu securitatea energetică",\n  "stance": "Opoziție suveranistă",\n  "sentiment": "Ostil",\n  "tone": "Populist, Agresiv",\n  "topic": "Securitate energetică, Suveranitate națională",\n  "interpretation_problem": "Comentariul promovează o viziune naționalistă și izolaționistă asupra securității energetice, ignorând interdependențele europene și soluțiile colective. Sugestia de a \'căuta sursele de petrol care le are România\' și de a cumpăra \'cât de mult putem depozita\' implică o abordare unilaterală, potențial dăunătoare pentru stabilitatea regională și pentru principiile solidarității europene în fața crizelor energetice. Limbajul este simplist și emoțional, sugerând o lipsă de încredere în capacitatea statului de a gestiona eficient resursele prin mecanisme europene."\n}\n```'

In [31]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [32]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,George Simion,"Opoziție suveranistă (implicit, prin asocierea...","Pozitiv (față de George Simion, dar cu subtext...","Populist, Naționalist","Naționalism, Suveranitate națională","Deși aparent un mesaj de felicitare, utilizare...",
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,AUR (Alianța pentru Unirea Românilor) și Georg...,Opoziție suveranistă,Pozitiv,Populist,Suveranitate națională,Comentariul exprimă sprijin necondiționat pent...,
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"George Simion, instituția prezidențială din Ro...",Opoziție suveranistă,Neutru spre ușor pozitiv,"Populist, Naționalist","Naționalism, Identitate națională, Leadership ...",Comentariul promovează o viziune naționalistă ...,
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Guvernul României / Instituțiile responsabile ...,Opoziție suveranistă,Ostil,"Populist, Agresiv","Securitate energetică, Suveranitate națională",Comentariul promovează o viziune naționalistă ...,
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"UE, Guvernul României, instituții democratice ...",Opoziție suveranistă,Extrem de ostil,"Populist, Agresiv, Conspiraționist","Suveranitate națională, anti-elită, naționalism","Comentariul promovează o retorică extremistă, ...",
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","Instituții democratice (Președinția României, ...",Opoziție suveranistă,Ostil,"Populist, Agresiv","Critica politică internă, Suveranitate naționa...",Comentariul utilizează un limbaj degradant și ...,
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George (un politician),Neutru,Pozitiv,Rațional,Interacțiunea politician-cetățean,Acest comentariu nu prezintă elemente problema...,
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"Uniunea Europeană (implicit, prin asocierea cu...",Opoziție suveranistă,Ostil,"Conspiraționist, Populist","Suveranitate națională, Manipulare politică",Comentariul utilizează o analogie istorică ana...,
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",AUR (Alianța pentru Unirea Românilor) și Georg...,Opoziție suveranistă,Extrem de pozitiv (în contextul autorului),"Populist, Emoțional","Naționalism, Suveranitate națională",Comentariul exprimă un sprijin fervent pentru ...,
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George (un politician),Neutru,Pozitiv,Susținător,Politica internă,Comentariul nu prezintă elemente suveraniste s...,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [42]:
results_df.to_csv("rezultate_c3_Tema1.csv", index=False, encoding='utf-8-sig')

# Promptul separă corect sentimentul general de poziționare față de target?


#### Promptul reușește să facă distincția tehnică între sentimentul autorului și poziționarea sa. Modul în care am construit promptul a fost foarte subiectiv, cerându-i modelului să adopte personajul unui „analist politic pro-european și ferm anti-suveranist”. În loc să extragă strict poziționarea obiectivă din comentariu, a adăugat bias. În definiția pentru stance, am cerut "poziția autorului față de valorile europene" în loc de poziția sa directă față de entitatea menționată în text (target). Modelul trebuie instruit să fie un observator neutru și să extragă poziția strict față de ținta evidentă din acel comentariu (care poate fi un politician local, nu neapărat UE).